# Dataset filtering

In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np

# spatial
import geopandas as gpd
import rioxarray as rxr

path_data_raw = Path("/home/rooda/Datasets")
wd = Path('/home/rooda/OneDrive/Projects/AndeanGC')
os.chdir(wd)


## Data

In [ ]:
# read data from previous processing step
AndeanGC_data     = pd.read_csv('dataset/AndeanGC_data_1950_2024.csv')
AndeanGC_metadata = pd.read_csv('dataset/AndeanGC_metadata.csv').set_index('gauge_id')
AndeanGC_shape    = gpd.read_file('dataset/AndeanGC_shape.gpkg').set_index('gauge_id')
AndeanGC_shape["basin_area"] = AndeanGC_shape.to_crs(32719).area / 1e6

## Basins selection

In [ ]:
# basins with glacier area > 0.1% (RGI v6.0)
rgi_version = "RGI60"
rgi = pd.concat([gpd.read_file(path_data_raw / "GLACIERS/OUTLINES/{}_16.shp".format(rgi_version)), 
                 gpd.read_file(path_data_raw / "GLACIERS/OUTLINES/{}_17.shp".format(rgi_version))])
rgi = rgi.rename(columns={"RGIId": "rgi_id"}).set_index("rgi_id")
rgi_union = rgi.union_all()

AndeanGC_shape = AndeanGC_shape[AndeanGC_shape.intersects(rgi_union)].copy()
AndeanGC_shape['glacier_area_RGI60'] = AndeanGC_shape.intersection(rgi_union).to_crs(epsg=32719).area / 1e6
AndeanGC_shape = AndeanGC_shape.fillna(0)  # fill NaN values with 0

# > 0.1% glacier area
AndeanGC_shape['glacier_area_RGI60'] = (AndeanGC_shape.glacier_area_RGI60 * 100 / AndeanGC_shape.basin_area)
AndeanGC_shape = AndeanGC_shape[AndeanGC_shape.glacier_area_RGI60 > 0.1]

AndeanGC_data = AndeanGC_data[AndeanGC_shape.index]
AndeanGC_metadata = AndeanGC_metadata.loc[AndeanGC_shape.index]
AndeanGC_metadata = pd.concat([AndeanGC_metadata, AndeanGC_shape[["basin_area", "glacier_area_RGI60"]]], axis=1)

In [ ]:
# basins with more than 1 year of data (365 days)
AndeanGC_metadata["days_w_data"] = (~AndeanGC_data.isnull()).sum()
AndeanGC_metadata = AndeanGC_metadata[AndeanGC_metadata.days_w_data > 365]

In [ ]:
# basins with "minimal" intervention
keywords_remove = "Embalse|Boca Toma|Central|Presa|Canal"
AndeanGC_metadata = AndeanGC_metadata[~AndeanGC_metadata["gauge_name"].str.contains(keywords_remove, case=False, na=False)]
AndeanGC_data = AndeanGC_data[AndeanGC_metadata.index]
AndeanGC_shape = AndeanGC_shape.loc[AndeanGC_metadata.index]

## Save

In [ ]:
AndeanGC_metadata.to_csv('dataset/AndeanGC_metadata.csv')
AndeanGC_data.to_csv('dataset/AndeanGC_data_1950_2024.csv')
AndeanGC_shape.to_file('dataset/AndeanGC_shape.gpkg')